# Download one raw H&E patch from each dataset

For the TFM defense slide 4 cross-institution stain composite. Grabs one patch each from LC25000, NCT-CRC-HE, Chaoyang, LungHist700 — all showing adenocarcinoma-ish tissue — then triggers browser downloads.

**Run once, download the 4 files, then reply to Claude with the paths on your Mac.** Should take under 2 minutes end-to-end.

If any cell errors, the fallback branch prints where it looked. Edit the paths near the top of the second cell if your Drive layout differs from `/content/drive/MyDrive/TFM/datasets/…`.

In [ ]:
!pip install -q kagglehub

In [ ]:
from pathlib import Path
import shutil, glob

from google.colab import drive, files
import kagglehub

# ---- edit these if your Drive layout differs ----
DRIVE_CHAOYANG = '/content/drive/MyDrive/TFM/datasets/Chaoyang'
DRIVE_LUNGHIST = '/content/drive/MyDrive/TFM/datasets/LungHist700'
# --------------------------------------------------

drive.mount('/content/drive')

OUT = Path('/content/samples')
OUT.mkdir(exist_ok=True)
print(f'Outputs will land in {OUT}\n')


def first_match(*patterns):
    """Return the first file matching any of the given glob patterns, or None."""
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return Path(hits[0])
    return None


def report(label, src, dst):
    if src is None:
        print(f'  ⚠ {label}: NOT FOUND — check paths / dataset layout')
        return None
    shutil.copy(str(src), str(dst))
    print(f'  ✓ {label}: {src.name} → {dst.name}')
    return dst


saved = []

# ---- LC25000 — colon_aca ----
print('LC25000 (via kagglehub) — colon adenocarcinoma:')
lc_root = Path(kagglehub.dataset_download(
    'andrewmvd/lung-and-colon-cancer-histopathological-images'))
src = first_match(
    f'{lc_root}/**/colon_image_sets/colon_aca/*.jpeg',
    f'{lc_root}/**/colon_aca/*.jpeg',
)
dst = report('LC25000', src, OUT / f'lc25000_colon_aca{src.suffix}' if src else OUT / 'lc25000_none')
if dst:
    saved.append(dst)

# ---- NCT-CRC-HE-100K/7K — TUM ----
# Try a few common kaggle slugs; skip on error.
print('\nNCT-CRC-HE (via kagglehub) — TUM (tumor epithelium, colon):')
nct_root = None
for slug in ['imrankhan77/nct-crc-he-100k', 'nikhilroxtomar/nct-crc-he-100k']:
    try:
        nct_root = Path(kagglehub.dataset_download(slug))
        print(f'  downloaded from {slug}')
        break
    except Exception as e:
        print(f'  {slug}: {type(e).__name__} {e}')
if nct_root is not None:
    src = first_match(
        f'{nct_root}/**/TUM/*.tif', f'{nct_root}/**/TUM/*.tiff',
        f'{nct_root}/**/TUM/*.jpg', f'{nct_root}/**/TUM/*.png',
    )
    dst = report('NCT-CRC', src, OUT / f'nct_crc_tum{src.suffix}' if src else OUT / 'nct_crc_none')
    if dst:
        saved.append(dst)

# ---- Chaoyang — cancer/adenocarcinoma ----
# Chaoyang naming: filenames often carry the class label (e.g. `_1.jpg` for cancer).
# Try labelled filenames first, then fall back to any patch.
print('\nChaoyang (from Drive) — adenocarcinoma / cancer:')
chaoyang_root = Path(DRIVE_CHAOYANG)
src = first_match(
    f'{chaoyang_root}/**/*_1.jpg', f'{chaoyang_root}/**/*_1.png',
    f'{chaoyang_root}/**/cancer/*.jpg', f'{chaoyang_root}/**/cancer/*.png',
    f'{chaoyang_root}/**/adenocarcinoma/*.jpg', f'{chaoyang_root}/**/adenocarcinoma/*.png',
    f'{chaoyang_root}/**/*.jpg', f'{chaoyang_root}/**/*.png',
)
dst = report('Chaoyang', src, OUT / f'chaoyang_cancer{src.suffix}' if src else OUT / 'chaoyang_none')
if dst:
    saved.append(dst)

# ---- LungHist700 — aca (lung adenocarcinoma), 20× ----
# LungHist700 filenames commonly encode class + magnification, e.g. `aca_bd_20x_…`.
print('\nLungHist700 (from Drive) — lung adenocarcinoma at 20×:')
lung_root = Path(DRIVE_LUNGHIST)
src = first_match(
    f'{lung_root}/**/aca*20x*.png', f'{lung_root}/**/aca*20x*.jpg', f'{lung_root}/**/aca*20x*.tif',
    f'{lung_root}/**/aca_*.png', f'{lung_root}/**/aca_*.jpg',
    f'{lung_root}/**/aca/*.png', f'{lung_root}/**/aca/*.jpg',
    f'{lung_root}/**/*.png', f'{lung_root}/**/*.jpg',
)
dst = report('LungHist700', src, OUT / f'lunghist700_aca{src.suffix}' if src else OUT / 'lunghist700_none')
if dst:
    saved.append(dst)

print('\n---')
print(f'Saved {len(saved)} sample(s) to {OUT}:')
for p in saved:
    print(f'  {p}')

In [ ]:
# Quick visual sanity check
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(saved), figsize=(4 * len(saved), 4))
if len(saved) == 1:
    axes = [axes]
for ax, p in zip(axes, saved):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Trigger browser downloads (one prompt per file)
for p in saved:
    files.download(str(p))